In [1]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_openai import ChatOpenAI
from os import getenv
from langgraph.checkpoint.memory import MemorySaver

In [2]:
from langgraph.graph.message import add_messages

class chatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [3]:
from pyexpat.errors import messages


llm = ChatOpenAI(
    api_key=getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="openai/gpt-oss-120b:free",
)

def chat_node(state: chatState):

    messages = state["messages"]   # always assign

    response = llm.invoke(messages)

    return {
        "messages": messages + [response]
    }

In [4]:
checkpoint = MemorySaver()
graph = StateGraph(chatState)

#add node 
graph.add_node("chat_node", chat_node)

#add edges
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpoint)

In [5]:
from langchain_core.messages import HumanMessage

# initial_state = {"messages": [HumanMessage(content="what is the real name of gandhi")]}

# result = chatbot.invoke(initial_state)
# print(result["messages"][-1].content)

In [6]:
thread_id = "1"

while True:

    user_message = input("enter a message: ")
    print("user: ", user_message)

    if user_message.strip().lower() in ["exit", "bye", "quit", "tata"]:
        break

    config = {"configurable": {"thread_id": thread_id}}

    response = chatbot.invoke(
        {"messages": HumanMessage(content=user_message)}, config=config
    )

    print("AI: ", response["messages"][-1].content)

user:  what is my name
AI:  I don’t actually know your name. If you’d like me to address you personally, just let me know what you’d like to be called!
user:  ankit
AI:  Nice to meet you, Ankit! If there’s anything specific you’d like help with, just let me know.
user:  now tell me my name
AI:  Your name is Ankit.
user:  quit
